[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C31_Coding_Agent_Course/01_file_tools/01_file_tools.ipynb)

# 01 · 文件工具（agent 的手）

目标：用**纯标准库**从零写出编码 agent 的文件工具——**带行号读**、**写**、**唯一匹配字符串替换编辑**、**路径安全校验**、**unified diff**——并在 `tempfile` 临时工作区里**真实读写**、用 `assert` 验证。

路线：带行号读 → 写(建新文件) → 字符串替换编辑(唯一匹配) → 路径安全 → unified diff → 把工具拼成「手」→ ✏️ 练习 → 📖 答案 → 🧪 真实 stdlib 源码胶囊。

> 心智模型：**读给 LLM 坐标系（行号），编辑用唯一字符串匹配（内容寻址比行号稳），改动可审计（diff），路径永远关在工作区里（安全）**。

## 0 · 准备：一个隔离的临时工作区

所有文件操作都在 `tempfile.mkdtemp()` 沙箱里**真实执行**，与你的系统隔离、用完即焚。

In [ ]:
import os, tempfile, shutil, difflib

WORK = tempfile.mkdtemp(prefix='c31_files_')
print('工作区:', WORK)

def seed(path, content):
    '''在工作区里真的写一个文件（仅用于铺设样例，不做安全校验）。'''
    full = os.path.join(WORK, path)
    os.makedirs(os.path.dirname(full), exist_ok=True) if os.path.dirname(path) else None
    with open(full, 'w', encoding='utf-8') as f:
        f.write(content)
    return full

seed('calc.py', 'def add(a, b):\n    return a - b   # BUG\n\n'
                 'def mul(a, b):\n    return a * b\n')
print('已铺设 calc.py ✅')

## 1 · 读文件：带行号 + 按范围

对 agent 友好的读：**带行号**（给 LLM 一个坐标系，方便后续指认「第几行」）、**支持行范围**（大文件不必整读、省 token）。

In [ ]:
def read_file(work, path, start=1, end=None):
    '''读文件，每行带【右对齐行号 + Tab + 原文】。start/end 为 1-based 行范围。'''
    full = os.path.join(work, path)
    lines = open(full, encoding='utf-8').read().splitlines()
    end = end or len(lines)
    out = []
    for i in range(start - 1, min(end, len(lines))):
        out.append(f'{i+1:6d}\t{lines[i]}')
    return '\n'.join(out)

view = read_file(WORK, 'calc.py')
print(view)
# 行号让我们能精确指认：bug 在第 2 行
assert view.splitlines()[1].endswith('return a - b   # BUG')
assert view.splitlines()[1].lstrip().startswith('2\t') or '2\t' in view.splitlines()[1]
# 按范围读：只看第 1-2 行
head = read_file(WORK, 'calc.py', start=1, end=2)
assert len(head.splitlines()) == 2
print('\n✅ 带行号读 + 范围读正确：LLM 拿到了带坐标的代码')

## 2 · 写文件：适合创建新文件

`write` 整体覆盖，**适合建新文件**（如新增一个测试）。改已有文件别用它（要 LLM 复述整份文件，长文件易漏抄→静默损坏）——那是第 3 节精确编辑的活。

我们顺手做**原子写**：先写临时文件再 `os.replace` 顶替，写到一半崩溃也不会留半截文件。

In [ ]:
def write_file(work, path, content):
    '''原子写：先写 .tmp 再 os.replace 改名顶替（要么旧的完整、要么新的完整）。'''
    full = os.path.join(work, path)
    tmp = full + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        f.write(content)
    os.replace(tmp, full)              # 原子改名
    return f'已写入 {path}（{len(content)} 字符）'

msg = write_file(WORK, 'test_calc.py',
                 'from calc import add, mul\n\n'
                 'def test_add():\n    assert add(2, 3) == 5\n\n'
                 'def test_mul():\n    assert mul(2, 3) == 6\n')
print(msg)
assert os.path.exists(os.path.join(WORK, 'test_calc.py'))
assert not os.path.exists(os.path.join(WORK, 'test_calc.py.tmp')), '临时文件应已被改名'
content = open(os.path.join(WORK, 'test_calc.py')).read()
assert 'def test_add' in content and 'def test_mul' in content
print('✅ 原子写成功：新建了测试文件，无残留 .tmp')

## 3 · 精确编辑：唯一匹配的字符串替换 ⭐

现代编码 agent 的主力编辑方式。给【旧串】和【新串】，**旧串必须在文件里恰好唯一匹配**才替换：
- 匹配 **0 次** → LLM 可能记错代码 → 拒绝；
- 匹配 **≥2 次** → 有歧义、会改错地方 → 拒绝，逼 LLM 给更多上下文；
- 恰好 **1 次** → 安全替换。

In [ ]:
def edit_file(work, path, old, new):
    '''唯一匹配的字符串替换编辑。旧串非唯一则拒绝并给可操作错误。'''
    full = os.path.join(work, path)
    text = open(full, encoding='utf-8').read()
    count = text.count(old)
    if count == 0:
        raise ValueError('旧字符串未找到：可能记错代码，拒绝编辑')
    if count > 1:
        raise ValueError(f'旧字符串匹配 {count} 处（有歧义）：请提供更多上下文使其唯一')
    new_text = text.replace(old, new)
    with open(full, 'w', encoding='utf-8') as f:
        f.write(new_text)
    return f'已编辑 {path}：替换 1 处'

# 修 bug：a - b -> a + b（这段在文件里唯一）
print(edit_file(WORK, 'calc.py', 'return a - b   # BUG', 'return a + b'))
assert 'return a + b' in open(os.path.join(WORK, 'calc.py')).read()
print('✅ 唯一匹配，bug 已修')

# 演示安全护栏：匹配 0 次 / 多次都被拒绝
for bad_old, why in [('return a - b', '已被改掉，0 次'), ('return', '出现多次，有歧义')]:
    try:
        edit_file(WORK, 'calc.py', bad_old, 'X'); raise AssertionError('本应拒绝')
    except ValueError as e:
        print(f'  正确拒绝 [{why}]: {e}')
print('✅ 非唯一匹配被安全拒绝（宁可拒绝，不可改错）')

## 4 · 路径安全：永远把 agent 关在工作区里

会写文件的 agent 若不校验路径，LLM 给个 `../../../etc/passwd` 就能读写系统文件（**path traversal**）。
防御：拼接后用 `os.path.realpath` 规范化（展开 `..` 与符号链接），再检查仍在工作区根之下。

In [ ]:
def safe_path(work, path):
    '''把 path 锚定到工作区并规范化；越界则拒绝。返回安全的绝对路径。'''
    work_root = os.path.realpath(work)
    target = os.path.realpath(os.path.join(work_root, path))
    if target != work_root and not target.startswith(work_root + os.sep):
        raise ValueError(f'路径越界，拒绝访问：{path}')
    return target

# 合法路径：通过
ok = safe_path(WORK, 'calc.py')
assert ok == os.path.join(os.path.realpath(WORK), 'calc.py')
assert ok.startswith(os.path.realpath(WORK))
print('合法路径通过:', os.path.basename(ok))

# 越界路径：全部拒绝
for evil in ['../escape.txt', '../../etc/passwd', '/etc/passwd', 'a/../../../../tmp/x']:
    try:
        safe_path(WORK, evil); raise AssertionError(f'本应拒绝 {evil}')
    except ValueError:
        print(f'  正确拒绝越界: {evil}')
print('✅ 路径穿越被挡住：agent 被关在工作区内')

## 5 · Unified diff：让每处改动可审计

`difflib.unified_diff` 产出 `git diff` 同款格式：`---/+++` 头 + `@@ -a,b +c,d @@` hunk + `-`删`+`增。
审阅 agent 的工作，看 diff 远比看整份文件高效。

In [ ]:
def make_diff(old_text, new_text, path):
    diff = difflib.unified_diff(
        old_text.splitlines(keepends=True),
        new_text.splitlines(keepends=True),
        fromfile=f'a/{path}', tofile=f'b/{path}')
    return ''.join(diff)

old = 'def add(a, b):\n    return a - b\n'
new = 'def add(a, b):\n    return a + b\n'
d = make_diff(old, new, 'calc.py')
print(d)
# 结构核对：有文件头、有 hunk 头、删了减法行、加了加法行
assert d.startswith('--- a/calc.py') and '+++ b/calc.py' in d
assert '@@' in d
assert '-    return a - b' in d and '+    return a + b' in d
print('✅ diff 一目了然：只显示改了什么，可供人审阅 / 应用 / 入库')

## 6 · 把四件工具拼成 agent 的「手」

读→编辑→审计 的闭环：每个工具内部先过 `safe_path`，编辑后自动产出 diff。下面封装一个带护栏、会返回 diff 的 `edit_and_diff`。

In [ ]:
def safe_read(work, path, start=1, end=None):
    full = safe_path(work, path)
    lines = open(full, encoding='utf-8').read().splitlines()
    end = end or len(lines)
    return '\n'.join(f'{i+1:6d}\t{lines[i]}' for i in range(start-1, min(end, len(lines))))

def edit_and_diff(work, path, old, new):
    '''带路径安全 + 唯一匹配 + 自动 diff 的编辑。返回 (消息, diff)。'''
    full = safe_path(work, path)                 # 护栏
    before = open(full, encoding='utf-8').read()
    cnt = before.count(old)
    if cnt != 1:
        raise ValueError(f'旧串匹配 {cnt} 处，需恰好 1 处')
    after = before.replace(old, new)
    with open(full, 'w', encoding='utf-8') as f:
        f.write(after)
    return f'已编辑 {path}', make_diff(before, after, path)

# 用它把 mul 改成支持默认参数（演示一次完整的读-改-审）
print('改前：'); print(safe_read(WORK, 'calc.py', 4, 5))
msg, d = edit_and_diff(WORK, 'calc.py', 'def mul(a, b):', 'def mul(a, b=1):')
print('\n' + msg + '，改动如下：\n' + d)
assert 'def mul(a, b=1):' in open(os.path.join(WORK,'calc.py')).read()
assert '-def mul(a, b):' in d and '+def mul(a, b=1):' in d
print('✅ 读→改→审 闭环跑通：安全、精确、可审计')

---
## ✏️ 练习 1：让字符串替换更稳健（去歧义提示）

增强 `edit_file`：当旧串匹配 **多处** 时，错误信息里要**报出每一处所在的行号**，帮 LLM 知道该补哪段上下文。

实现 `edit_robust(work, path, old, new)`：唯一则替换并返回消息；0 次报错；多次则 `raise ValueError`，消息形如 `匹配 N 处，分别在行: [r1, r2, ...]`。

In [ ]:
def edit_robust(work, path, old, new):
    full = os.path.join(work, path)
    text = open(full, encoding='utf-8').read()
    # TODO:
    #  1) cnt = text.count(old)
    #  2) cnt==0 -> raise ValueError('旧字符串未找到')
    #  3) cnt>1  -> 找出每个匹配所在行号(1-based)，raise ValueError(f'匹配 {cnt} 处，分别在行: {rows}')
    #       提示：用 text.find(old, pos) 循环找所有起始位置；行号 = text.count('\n', 0, pos)+1
    #  4) 恰好 1 处 -> 写回 text.replace(old,new)，return f'已编辑 {path}：替换 1 处'
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
seed('ex1.py', 'x = 1\ny = 1\nz = 2\nw = 1\n')   # '= 1' 出现 3 次
# 多处匹配应报出行号 1,2,4
try:
    edit_robust(WORK, 'ex1.py', '= 1', '= 99'); raise AssertionError('本应拒绝')
except ValueError as e:
    msg = str(e)
    print('多处匹配的错误:', msg)
    assert '3' in msg and '1' in msg and '2' in msg and '4' in msg, '应报出 3 处及行号 1,2,4'
# 唯一匹配应成功
out = edit_robust(WORK, 'ex1.py', 'z = 2', 'z = 200')
assert 'z = 200' in open(os.path.join(WORK,'ex1.py')).read()
# 0 次匹配应报错
try:
    edit_robust(WORK, 'ex1.py', 'nope', 'x'); raise AssertionError('本应拒绝')
except ValueError:
    pass
print('✅ 练习 1 通过：多处匹配能精确报出各行号，引导 LLM 补上下文')

## ✏️ 练习 2：按行范围替换（insert / delete / replace 三合一）

有时按行号操作更自然（如「删掉 3-5 行」「在第 2 行后插入」）。实现一个**行范围替换** `replace_lines`：

`replace_lines(work, path, start, end, new_lines)`：把 1-based 闭区间 `[start, end]` 的行替换成 `new_lines`（一个字符串列表）。
- `new_lines=[]` → 等价删除这些行；
- `start=end+1`（空区间）→ 等价在 start 处之前插入。返回新文件总行数。

In [ ]:
def replace_lines(work, path, start, end, new_lines):
    full = os.path.join(work, path)
    lines = open(full, encoding='utf-8').read().splitlines()
    # TODO: 把 lines[start-1:end] 这段替换成 new_lines（注意 1-based -> 0-based 切片）
    #       新内容 = lines[:start-1] + new_lines + lines[end:]
    #       写回（每行加 '\n'），返回新行数
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
seed('ex2.py', 'L1\nL2\nL3\nL4\nL5\n')
# replace：把 2-3 行换成一行 'NEW'
n = replace_lines(WORK, 'ex2.py', 2, 3, ['NEW'])
got = open(os.path.join(WORK,'ex2.py')).read().splitlines()
assert got == ['L1','NEW','L4','L5'] and n == 4, got
# delete：删掉第 1 行（new_lines=[]）
replace_lines(WORK, 'ex2.py', 1, 1, [])
got = open(os.path.join(WORK,'ex2.py')).read().splitlines()
assert got == ['NEW','L4','L5'], got
# insert：在第 2 行之前插入（空区间 start=2,end=1）
replace_lines(WORK, 'ex2.py', 2, 1, ['INS'])
got = open(os.path.join(WORK,'ex2.py')).read().splitlines()
assert got == ['NEW','INS','L4','L5'], got
print('✅ 练习 2 通过：replace / delete / insert 三种行操作都对')

## ✏️ 练习 3：更严的路径校验（带符号链接陷阱）

实现 `is_inside(work, path)`：返回 `path` 锚定到工作区、规范化后**是否仍在工作区内**（布尔，不抛异常）。

要求能挡住：`..` 穿越、绝对路径、以及**工作区内指向外部的符号链接**（用 `realpath` 跟随链接后判断）。

In [ ]:
def is_inside(work, path):
    # TODO: 仿照 safe_path，但返回 True/False 而非抛异常
    #   root = os.path.realpath(work)
    #   target = os.path.realpath(os.path.join(root, path))
    #   return target == root or target.startswith(root + os.sep)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert is_inside(WORK, 'calc.py') is True
assert is_inside(WORK, 'sub/dir/x.py') is True        # 还不存在也应判为「在内」
assert is_inside(WORK, '../outside.py') is False
assert is_inside(WORK, '/etc/passwd') is False
# 符号链接陷阱：在工作区内造一个指向外部 /tmp 的软链
ext = tempfile.mkdtemp(prefix='c31_external_')
try:
    link = os.path.join(WORK, 'sneaky')
    try:
        os.symlink(ext, link)
        # 经由软链访问外部文件，realpath 跟随后应判为越界
        assert is_inside(WORK, 'sneaky/secret.txt') is False, '软链越界应被识破'
        print('  符号链接越界被识破 ✅')
    except (OSError, NotImplementedError) as e:
        if isinstance(e, NotImplementedError): raise
        print('  (本平台不支持创建符号链接，跳过该子项)')
finally:
    shutil.rmtree(ext, ignore_errors=True)
print('✅ 练习 3 通过：.. / 绝对路径 / 符号链接越界都能识别')

## ✏️ 练习 4：从 diff 数出改了几行

解析 unified diff 文本，统计**新增行数**与**删除行数**（用于给人一个「改动规模」的概览）。

实现 `diff_stat(diff_text)` → 返回 `(added, removed)`。注意：以 `+` 开头但**不是** `+++` 文件头的行才算新增；`-` 同理（排除 `---`）。

In [ ]:
def diff_stat(diff_text):
    # TODO: 遍历每一行：
    #   startswith('+') and not startswith('+++') -> added += 1
    #   startswith('-') and not startswith('---') -> removed += 1
    #   返回 (added, removed)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
old = 'a\nb\nc\nd\n'
new = 'a\nB\nC\nd\ne\n'       # 改了 b->B, c->C，加了 e
d = make_diff(old, new, 'f.py')
added, removed = diff_stat(d)
print(f'diff 统计：+{added} 行 / -{removed} 行')
assert (added, removed) == (3, 2), (added, removed)   # +B +C +e / -b -c
# 纯新增文件
d2 = make_diff('', 'x\ny\n', 'new.py')
a2, r2 = diff_stat(d2)
assert a2 == 2 and r2 == 0
print('✅ 练习 4 通过：能从 diff 数出改动规模，且不误把 +++/--- 头算进去')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def edit_robust(work, path, old, new):
    full = os.path.join(work, path)
    text = open(full, encoding='utf-8').read()
    cnt = text.count(old)
    if cnt == 0:
        raise ValueError('旧字符串未找到')
    if cnt > 1:
        rows, pos = [], text.find(old)
        while pos != -1:
            rows.append(text.count('\n', 0, pos) + 1)
            pos = text.find(old, pos + 1)
        raise ValueError(f'匹配 {cnt} 处，分别在行: {rows}')
    with open(full, 'w', encoding='utf-8') as f:
        f.write(text.replace(old, new))
    return f'已编辑 {path}：替换 1 处'

In [ ]:
# 练习 2 参考答案
def replace_lines(work, path, start, end, new_lines):
    full = os.path.join(work, path)
    lines = open(full, encoding='utf-8').read().splitlines()
    updated = lines[:start-1] + list(new_lines) + lines[end:]
    with open(full, 'w', encoding='utf-8') as f:
        f.write('\n'.join(updated) + ('\n' if updated else ''))
    return len(updated)

In [ ]:
# 练习 3 参考答案
def is_inside(work, path):
    root = os.path.realpath(work)
    target = os.path.realpath(os.path.join(root, path))
    return target == root or target.startswith(root + os.sep)

In [ ]:
# 练习 4 参考答案
def diff_stat(diff_text):
    added = removed = 0
    for ln in diff_text.splitlines():
        if ln.startswith('+') and not ln.startswith('+++'):
            added += 1
        elif ln.startswith('-') and not ln.startswith('---'):
            removed += 1
    return added, removed

---
## 🧪 真实数据胶囊：在一份真实 stdlib 源码上跑文件工具

用 Python **标准库里一个真实模块的源码**（`textwrap.py`，随 Python 一起安装、真实存在）当样本，把它复制进工作区，用我们的工具**真实地**读它、对它做一次唯一匹配编辑、并生成 diff。

（无需联网：源码就在你本机的 Python 安装里，用 `inspect.getsourcefile` 定位。）

In [ ]:
import inspect, textwrap as _tw
src_path = inspect.getsourcefile(_tw)         # 本机真实存在的 stdlib 源码路径
real_src = open(src_path, encoding='utf-8').read()
print('真实源码:', os.path.basename(src_path), f'({len(real_src)} 字符, {real_src.count(chr(10))} 行)')
# 复制进工作区，全程用我们的工具操作它
write_file(WORK, 'textwrap_copy.py', real_src)
view = read_file(WORK, 'textwrap_copy.py', start=1, end=3)
print('带行号读前 3 行:')
print(view)
assert len(view.splitlines()) == 3
# 找一个在文件里唯一出现的字符串来安全编辑：模块的 __all__ 定义通常唯一
assert real_src.count("__all__ = ") == 1, '该样本里 __all__ 应唯一，便于唯一匹配编辑'
print('\n✅ 胶囊准备就绪：真实 stdlib 源码已在工作区，可被工具安全操作')

**🧪 胶囊练习**：在这份真实源码副本上，用 `edit_robust` 把 `__all__ = ` 这一处改写（在其后加一个注释标记），再用 `make_diff` 产出改动，并用 `diff_stat` 确认恰好 **+? / -?** 行变化。补全下面骨架。

In [ ]:
# 找到 __all__ 那一整行（唯一），在行尾追加注释
lines = real_src.splitlines()
all_line = next(ln for ln in lines if ln.startswith('__all__ = '))
old_line = all_line
new_line = all_line + '  # edited by agent'
before = open(os.path.join(WORK, 'textwrap_copy.py'), encoding='utf-8').read()
# TODO: 用 edit_robust 把 old_line 换成 new_line（它在文件里唯一）
#       然后 after = 重新读文件；d = make_diff(before, after, 'textwrap_copy.py')
#       added, removed = diff_stat(d)
raise NotImplementedError

In [ ]:
# 自测
assert 'edited by agent' in open(os.path.join(WORK,'textwrap_copy.py'), encoding='utf-8').read()
assert (added, removed) == (1, 1), '改一整行 = +1/-1'
print('真实源码上的一次安全编辑：')
print(d)
print('✅ 胶囊练习通过：在真实 stdlib 源码上完成了 读→唯一匹配编辑→diff 的闭环')

In [ ]:
# 📖 胶囊参考答案
msg = edit_robust(WORK, 'textwrap_copy.py', old_line, new_line)
after = open(os.path.join(WORK, 'textwrap_copy.py'), encoding='utf-8').read()
d = make_diff(before, after, 'textwrap_copy.py')
added, removed = diff_stat(d)
print(msg, '|', (added, removed))

---
## 🔧 旁注：这些工具如何暴露给真实 Claude

把文件工具接到真实模型时，要给每个工具写一份 **tool schema**（Anthropic `tools=[...]` 的元素），告诉 Claude 工具叫什么、干什么、要哪些参数。下面是 `edit_file` 的 schema 形态（**本环境不调用**）：

```python
EDIT_TOOL_SCHEMA = {
    'name': 'edit_file',
    'description': '在文件中把【唯一匹配】的 old 字符串替换为 new。old 必须在文件中恰好出现一次。',
    'input_schema': {
        'type': 'object',
        'properties': {
            'path': {'type': 'string', 'description': '相对工作区的文件路径'},
            'old':  {'type': 'string', 'description': '要被替换的原文片段（需唯一）'},
            'new':  {'type': 'string', 'description': '替换后的新内容'},
        },
        'required': ['path', 'old', 'new'],
    },
}
```

Claude 会据此 `description` 决定何时调用、如何填参；返回的 `tool_use` 块里 `input` 就是 `{path, old, new}`。完整的工具往返循环在**模块 05**。这里要记住的是：**工具的 description 是写给 LLM 的提示词**——写清「old 必须唯一」能让模型少犯错。

In [ ]:
# 清理工作区
shutil.rmtree(WORK, ignore_errors=True)
print('工作区已清理 ✅')

### 小结
- **读**：带行号 + 按范围 → 给 LLM 一个可寻址的坐标系（报错定位、编辑锚点都靠它）。
- **写**：整体覆盖，**只用于建新文件**；改已有文件易静默损坏。原子写防半截文件。
- **编辑**：**唯一匹配的字符串替换**是主力——内容寻址比行号稳；非唯一一律拒绝（宁拒不错）。
- **路径安全**：`realpath` + 前缀检查，永远把 agent 关在工作区里；不信任任何来自 LLM 的路径。
- **diff**：`difflib.unified_diff` 让每处改动可审计、可应用、可入库——agent 可信的前提。

下一站：**模块 02 · Shell 工具** —— 给 agent 装上脚，安全地跑命令与测试。